# Serviços financeiros — Medallion e Spark ML no AIDP
Adaptado de https://github.com/oracle-samples/oracle-aidp-samples/blob/main/data-engineering/transformation/medallion-lake/financial_services_medallion_architecture_demo.ipynb

Dados inteiramente sintéticos; nenhuma credencial ou dado financeiro real é usado.
Cria quatro tabelas Delta gerenciadas em três schemas exclusivos no catálogo Standard `default`.
Mantém as 5.000 contas, 10–50 transações por conta e Random Forest de 100 árvores do exemplo.
Liquid clustering fica desabilitado por padrão e não é condição de sucesso; pode ser solicitado com ENABLE_LIQUID_CLUSTERING=true e exige suporte no runtime.
A marcação is_fraud é derivada de um risk_score aleatório, como no original: as métricas demonstram o pipeline, não validam um detector de fraude real. O risk_score não entra nas features.

In [ ]:
import json, math, random, re
from datetime import datetime, timedelta
from decimal import Decimal
from uuid import uuid4
import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DecimalType, IntegerType

if 'spark' not in globals() or 'oidlUtils' not in globals():
    raise RuntimeError('Execute no runtime AIDP com compute anexado.')

def parameter(name, default):
    value = oidlUtils.parameters.getParameter(name, str(default))
    return str(default) if value is None else str(value).strip()

def ident(s):
    if not re.fullmatch(r'[A-Za-z_][A-Za-z0-9_]*', s):
        raise ValueError(f'Identificador inválido: {s!r}')
    return f'`{s}`'

catalog = parameter('CATALOG_NAME', 'default')
accounts = int(parameter('ACCOUNT_COUNT', 5000))
seed = int(parameter('SEED', 42))
num_trees = int(parameter('RF_NUM_TREES', 100))
max_depth = int(parameter('RF_MAX_DEPTH', 10))
liquid = parameter('ENABLE_LIQUID_CLUSTERING', 'false').lower() == 'true'
if not 100 <= accounts <= 5000 or not 1 <= num_trees <= 100 or not 1 <= max_depth <= 10:
    raise ValueError('Limites do demo: 100–5000 contas, 1–100 árvores, profundidade 1–10.')
run_id = uuid4().hex[:12]
schemas = {layer: f'finance_{layer}_{run_id}' for layer in ('bronze', 'silver', 'gold')}
def table(layer, name):
    return '.'.join(ident(v) for v in (catalog, schemas[layer], name))
bronze_table = table('bronze', 'account_transactions_bronze')
silver_table = table('silver', 'account_transactions_silver')
analytics_table = table('gold', 'transaction_analytics_gold')
features_table = table('gold', 'fraud_detection_model_gold')
result = {'run_id': run_id, 'catalog': catalog, 'schemas': schemas, 'accounts': accounts, 'seed': seed, 'liquid_clustering_enabled': liquid}
for layer, name in schemas.items():
    spark.sql(f'CREATE SCHEMA {ident(catalog)}.{ident(name)}')
print('FINANCE_STAGE=schemas_created ' + json.dumps(result))

In [ ]:
# Geração determinística de transações e schema explícito, inclusive Decimal e timestamp.
rng = random.Random(seed)
transaction_types = ['Deposit', 'Withdrawal', 'Transfer', 'Payment', 'ATM']
merchants = ['Retail', 'Restaurant', 'Online', 'Utilities', 'Entertainment', 'Groceries', 'Healthcare', 'Transportation']
locations = ['New York, NY', 'Los Angeles, CA', 'Chicago, IL', 'Houston, TX', 'Miami, FL', 'Online', 'ATM']
base_date = datetime(2024, 1, 1)
ingestion_time = datetime.utcnow()
records = []
for account_num in range(1, accounts + 1):
    for _ in range(rng.randint(10, 50)):
        date = base_date + timedelta(days=rng.randint(0, 365), hours=rng.randint(0, 23))
        kind = rng.choice(transaction_types)
        low, high, sign = (100, 10000, 1) if kind in ('Deposit', 'Transfer') else ((20, 500, -1) if kind == 'ATM' else (10, 2000, -1))
        amount = Decimal(str(round(rng.uniform(low, high) * sign, 2))).quantize(Decimal('0.01'))
        merchant = rng.choice(merchants)
        location = 'ATM' if kind == 'ATM' else ('Online' if merchant == 'Online' else rng.choice(locations))
        records.append((f'ACC{account_num:08d}', date, kind, amount, merchant, location, rng.randint(0, 100), ingestion_time))
raw_schema = StructType([
    StructField('account_id', StringType()), StructField('transaction_date', TimestampType()),
    StructField('transaction_type', StringType()), StructField('amount', DecimalType(15, 2)),
    StructField('merchant_category', StringType()), StructField('location', StringType()),
    StructField('risk_score', IntegerType()), StructField('ingestion_timestamp', TimestampType())
])
raw = spark.createDataFrame(records, raw_schema)
expected_rows = len(records)
# Todos os destinos são exclusivos desta execução; errorifexists evita sobrescritas.
def write_delta(frame, target, keys):
    if liquid:
        view = 'finance_stage_' + uuid4().hex[:8]
        frame.createOrReplaceTempView(view)
        spark.sql(f'CREATE TABLE {target} USING DELTA CLUSTER BY ({keys}) AS SELECT * FROM {view}')
        spark.catalog.dropTempView(view)
    else:
        frame.write.format('delta').mode('errorifexists').saveAsTable(target)
write_delta(raw, bronze_table, 'account_id, transaction_date')
bronze = spark.table(bronze_table)
bronze_count = bronze.count()
assert bronze_count == expected_rows
result['bronze_rows'] = bronze_count
print(f'FINANCE_STAGE=bronze_complete rows={bronze_count}')

In [ ]:
# Silver: validação, features temporais e timestamps coerentes com o schema gravado.
silver = (bronze
    .withColumn('month', F.month('transaction_date'))
    .withColumn('day_of_week', F.dayofweek('transaction_date'))
    .withColumn('hour', F.hour('transaction_date'))
    .withColumn('is_valid', F.col('amount').isNotNull() & F.col('account_id').isNotNull() & F.col('transaction_date').isNotNull() & F.col('risk_score').between(0, 100))
    .withColumn('data_quality_score', F.when(F.col('is_valid'), (F.lit(1.0) - F.col('risk_score') / 100.0) * 0.7 + 0.3).otherwise(0.0))
    .withColumn('processed_timestamp', F.current_timestamp())
    .filter('is_valid'))
write_delta(silver, silver_table, 'account_id, transaction_date')
silver = spark.table(silver_table)
silver_count = silver.count()
assert silver_count == expected_rows
assert silver.filter(~F.col('is_valid') | F.col('data_quality_score').isNull() | ~F.col('data_quality_score').between(0.3, 1.0)).count() == 0
result['silver_rows'] = silver_count
result['rejected_rows'] = bronze_count - silver_count
print(f'FINANCE_STAGE=silver_complete rows={silver_count}')

In [ ]:
# Gold: métricas mensais e distribuições de tipos de transação e estabelecimentos.
monthly = silver.withColumn('month_year', F.date_format('transaction_date', 'yyyy-MM'))
analytics = monthly.groupBy('account_id', 'month_year').agg(
    F.count('*').alias('total_transactions'), F.sum('amount').alias('total_amount'),
    F.avg('amount').alias('avg_amount'), F.avg('risk_score').alias('avg_risk_score'),
    F.sum(F.when(F.col('risk_score') > 60, 1).otherwise(0)).alias('high_risk_transactions'))
def distribution(column, output):
    return (monthly.groupBy('account_id', 'month_year', column).count()
        .groupBy('account_id', 'month_year')
        .agg(F.map_from_entries(F.collect_list(F.struct(F.col(column), F.col('count')))).alias(output)))
analytics = (analytics.join(distribution('transaction_type', 'transaction_types'), ['account_id', 'month_year'])
    .join(distribution('merchant_category', 'merchant_categories'), ['account_id', 'month_year'])
    .withColumn('created_timestamp', F.current_timestamp()))
write_delta(analytics, analytics_table, 'account_id, month_year')
gold_analytics = spark.table(analytics_table)
reconciled = gold_analytics.agg(F.sum('total_transactions').alias('transactions'), F.sum('total_amount').alias('amount')).first()
source_total = silver.agg(F.sum('amount').alias('amount')).first().amount
assert reconciled.transactions == silver_count
assert reconciled.amount == source_total
result['analytics_rows'] = gold_analytics.count()
result['amount_reconciled'] = True
ml_features = (silver.withColumn('is_fraud', F.when(F.col('risk_score') > 60, 1).otherwise(0))
    .select('account_id','transaction_date','transaction_type','amount','merchant_category','location','risk_score','month','day_of_week','hour','is_fraud')
    .withColumn('created_timestamp', F.current_timestamp()))
write_delta(ml_features, features_table, 'account_id, transaction_date')
ml_features = spark.table(features_table)
assert ml_features.count() == silver_count
result['ml_feature_rows'] = silver_count
print('FINANCE_STAGE=gold_complete ' + json.dumps(result, default=str))

## Treinamento e avaliação
Random Forest usa valor, tempo e categorias; `risk_score` e `is_fraud` não entram no vetor de features.
Categorias desconhecidas são mantidas pelo StringIndexer. O modelo é treinado apenas no split de treino.
O rótulo aleatório não representa fraude comprovada. AUC perto de 0,5 é plausível e não constitui falha do pipeline.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

ml_input = ml_features.withColumn('amount_ml', F.col('amount').cast('double'))
train_data, test_data = ml_input.randomSplit([0.8, 0.2], seed=seed)
train_data, test_data = train_data.cache(), test_data.cache()
train_count, test_count = train_data.count(), test_data.count()
if train_count == 0 or test_count == 0 or train_data.select('is_fraud').distinct().count() != 2 or test_data.select('is_fraud').distinct().count() != 2:
    raise ValueError('Treino e teste devem ser não vazios e conter ambas as classes.')
indexers = [StringIndexer(inputCol=c, outputCol=c + '_index', handleInvalid='keep') for c in ('transaction_type', 'merchant_category', 'location')]
feature_names = ['amount_ml','month','day_of_week','hour','transaction_type_index','merchant_category_index','location_index']
assembler = VectorAssembler(inputCols=feature_names, outputCol='features')
scaler = StandardScaler(inputCol='features', outputCol='scaled_features')
rf = RandomForestClassifier(labelCol='is_fraud', featuresCol='scaled_features', numTrees=num_trees, maxDepth=max_depth, seed=seed)
pipeline = Pipeline(stages=indexers + [assembler, scaler, rf])
print(f'FINANCE_STAGE=training_started train={train_count} test={test_count} trees={num_trees}')
model = pipeline.fit(train_data)
predictions = model.transform(test_data).cache()
auc = BinaryClassificationEvaluator(labelCol='is_fraud', rawPredictionCol='rawPrediction', metricName='areaUnderROC').evaluate(predictions)
assert math.isfinite(auc) and 0 <= auc <= 1
assert predictions.count() == test_count
confusion = {(int(r.is_fraud), int(r.prediction)): r['count'] for r in predictions.groupBy('is_fraud','prediction').count().collect()}
tp, tn = confusion.get((1,1),0), confusion.get((0,0),0)
fp, fn = confusion.get((0,1),0), confusion.get((1,0),0)
result.update(train_rows=train_count, test_rows=test_count, rf_num_trees=num_trees, rf_max_depth=max_depth, auc=float(auc),
    accuracy=(tp+tn)/test_count, precision=tp/(tp+fp) if tp+fp else 0.0, recall=tp/(tp+fn) if tp+fn else 0.0,
    confusion_matrix={f'actual_{a}_pred_{p}':count for (a,p),count in confusion.items()},
    feature_importance={name:float(value) for name,value in zip(feature_names, model.stages[-1].featureImportances)})
print('FINANCE_STAGE=training_complete ' + json.dumps(result, default=str))

In [ ]:
# Consultas de exemplo e evidência final das quatro tabelas persistidas.
for target in (bronze_table, silver_table, analytics_table, features_table):
    print('TABLE=' + target)
    spark.table(target).filter(F.col('account_id') == 'ACC00000001').limit(3).show(truncate=False)
result['tables'] = [t.replace('`','') for t in (bronze_table,silver_table,analytics_table,features_table)]
result['synthetic_labels_only'] = True
result['status'] = 'SUCCESS'
print('AIDP_FINANCE_MEDALLION_TEST_RESULT=' + json.dumps(result, sort_keys=True))
predictions.unpersist()
train_data.unpersist()
test_data.unpersist()

## Retenção
Os schemas e as quatro tabelas gerenciadas permanecem para inspeção. Não há DROP nem sobrescrita de tabelas existentes.
O modelo treinado permanece apenas na sessão desta execução; não foi publicado como serviço ou modelo produtivo.